# Load dữ liệu

In [3]:
import pandas as pd
import numpy as np
import re
import unicodedata
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, PredefinedSplit
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

# --- 1. CÁC HÀM PREPROCESSING (GIỮ NGUYÊN) ---
def clean_review_basic(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"\d+", " ", text)
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    text = text.replace("_", " ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

def remove_vietnamese_accents(text):
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize('NFD', text)
    text = ''.join(ch for ch in text if unicodedata.category(ch) != 'Mn')
    text = unicodedata.normalize('NFC', text)
    return text

def preprocess_review(text):
    text = clean_review_basic(text)
    text = remove_vietnamese_accents(text)
    return text

# --- 2. LOAD DATA TỪ 3 FILE ---
# Định nghĩa đường dẫn file
train_path = r"preprocessed-dataset\train_processed.json"
dev_path   = r"preprocessed-dataset\dev_processed.json"
test_path  = r"preprocessed-dataset\test_processed.json"

def load_and_process(path):
    """Hàm hỗ trợ đọc file và preprocess"""
    try:
        df = pd.read_json(path)
    except ValueError:
        df = pd.read_json(path, lines=True)
    
    # Áp dụng preprocessing
    df["review_clean"] = df["review"].apply(preprocess_review)
    # Chuẩn hóa nhãn về chữ thường
    df["sentiment"] = df["sentiment"].str.lower().str.strip()
    return df

print("Đang load dữ liệu...")
df_train = load_and_process(train_path)
df_dev   = load_and_process(dev_path)
df_test  = load_and_process(test_path)

print(f"Số lượng mẫu: Train={len(df_train)}, Dev={len(df_dev)}, Test={len(df_test)}")

# --- 3. ENCODE NHÃN (LABEL ENCODING) ---
le = LabelEncoder()
# Chỉ fit trên tập Train để đảm bảo tính khách quan
df_train["label_id"] = le.fit_transform(df_train["sentiment"])

# Transform cho Dev và Test (nếu gặp nhãn lạ sẽ lỗi, nhưng data chuẩn thường không sao)
df_dev["label_id"]   = le.transform(df_dev["sentiment"])
df_test["label_id"]  = le.transform(df_test["sentiment"])

print("Classes:", le.classes_)

# Tách features và labels ra mảng numpy
X_train = df_train["review_clean"].values
y_train = df_train["label_id"].values

X_dev   = df_dev["review_clean"].values
y_dev   = df_dev["label_id"].values

X_test  = df_test["review_clean"].values
y_test  = df_test["label_id"].values

Đang load dữ liệu...
Số lượng mẫu: Train=1710, Dev=244, Test=490
Classes: ['negative' 'neutral' 'positive']


# Hàm preprocessing: giữ như trước + bỏ dấu tiếng Việt

In [4]:
import re
import unicodedata

def clean_review_basic(text):
    """Tiền xử lý cơ bản như bạn yêu cầu: 
    lower, bỏ số, bỏ ký hiệu, thu gọn khoảng trắng
    """
    if not isinstance(text, str):
        return ""

    # viết thường
    text = text.lower()

    # bỏ số
    text = re.sub(r"\d+", " ", text)

    # bỏ ký hiệu, dấu câu (giữ lại chữ, số, khoảng trắng)
    # \w = chữ + số + _ ; \s = khoảng trắng
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)

    # bỏ riêng dấu gạch dưới nếu còn
    text = text.replace("_", " ")

    # thu gọn khoảng trắng
    text = re.sub(r"\s+", " ", text).strip()

    return text


def remove_vietnamese_accents(text):
    """Bỏ dấu tiếng Việt bằng unicodedata"""
    if not isinstance(text, str):
        return ""
    text = unicodedata.normalize('NFD', text)
    text = ''.join(ch for ch in text if unicodedata.category(ch) != 'Mn')
    text = unicodedata.normalize('NFC', text)
    return text


def preprocess_review(text):
    # bước 1: clean cơ bản
    text = clean_review_basic(text)
    # bước 2: bỏ dấu tiếng Việt
    text = remove_vietnamese_accents(text)
    return text

# Chuẩn hóa nhãn & encode label

# Chia train / test (và optionally valid)

In [ ]:
pip install xgboost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


# Vector hóa text bằng TF-IDF

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from xgboost import XGBClassifier

num_classes = len(df["label_id"].unique())

xgb_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.9,
        max_features=None
    )),
    ("xgb", XGBClassifier(
        objective="multi:softprob",
        num_class=num_classes,
        eval_metric="mlogloss",
        tree_method="hist",
        n_jobs=-1,
        # GIỮ CỐ ĐỊNH các tham số dưới đây
        n_estimators=300,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        reg_alpha=0.0
    ))
])

In [6]:
# --- 4. CẤU HÌNH PREDEFINED SPLIT ---
# Gộp Train và Dev lại thành một tập lớn để đưa vào hàm fit
X_combined = np.concatenate((X_train, X_dev), axis=0)
y_combined = np.concatenate((y_train, y_dev), axis=0)

# Tạo mảng chỉ mục (test_fold):
# -1: Mẫu thuộc tập Train (dùng để huấn luyện)
#  0: Mẫu thuộc tập Validation (dùng để chấm điểm Hyperparameters)
split_index = [-1] * len(X_train) + [0] * len(X_dev)

ps = PredefinedSplit(test_fold=split_index)

print("Đã tạo PredefinedSplit. GridSearch sẽ tune trên tập Dev cố định.")

Đã tạo PredefinedSplit. GridSearch sẽ tune trên tập Dev cố định.


## KNN


In [7]:
# --- 5. MODEL 1: KNN ---
knn_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_df=0.9)),
    ("knn", KNeighborsClassifier())
])

param_grid_knn = {
    "knn__n_neighbors": [3, 5, 7, 9, 15],
    "knn__weights": ["uniform", "distance"],
    "knn__metric": ["cosine", "euclidean"]
}

print("\n--- Đang chạy GridSearch cho KNN ---")
grid_knn = GridSearchCV(
    knn_pipeline,
    param_grid_knn,
    scoring="f1_macro",
    cv=ps,  # <--- Dùng split đã tạo ở trên
    n_jobs=-1,
    verbose=1
)

grid_knn.fit(X_combined, y_combined)

print("Best KNN params:", grid_knn.best_params_)
print("Best Dev F1-macro:", grid_knn.best_score_)

# Đánh giá trên tập Test
print("\n--- Kết quả KNN trên tập Test ---")
knn_best = grid_knn.best_estimator_
y_pred_knn = knn_best.predict(X_test)
print(classification_report(y_test, y_pred_knn, target_names=le.classes_))


--- Đang chạy GridSearch cho KNN ---
Fitting 1 folds for each of 20 candidates, totalling 20 fits
Best KNN params: {'knn__metric': 'cosine', 'knn__n_neighbors': 15, 'knn__weights': 'uniform'}
Best Dev F1-macro: 0.6359600380277995

--- Kết quả KNN trên tập Test ---
              precision    recall  f1-score   support

    negative       0.67      0.77      0.72       151
     neutral       0.66      0.69      0.68       194
    positive       0.66      0.52      0.58       145

    accuracy                           0.66       490
   macro avg       0.66      0.66      0.66       490
weighted avg       0.66      0.66      0.66       490



## Random Forest


In [8]:
# --- 6. MODEL 2: RANDOM FOREST ---
rf_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_df=0.9)),
    ("rf", RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1))
])

param_grid_rf = {
    "rf__max_depth": [None, 20, 50],
    "rf__min_samples_split": [2, 10],
    "rf__min_samples_leaf": [1, 4]
}

print("\n--- Đang chạy GridSearch cho Random Forest ---")
grid_rf = GridSearchCV(
    rf_pipeline,
    param_grid_rf,
    scoring="f1_macro",
    cv=ps, # <--- Dùng split đã tạo ở trên
    n_jobs=-1,
    verbose=1
)

grid_rf.fit(X_combined, y_combined)

print("Best RF params:", grid_rf.best_params_)
print("Best Dev F1-macro:", grid_rf.best_score_)

# Đánh giá trên tập Test
print("\n--- Kết quả Random Forest trên tập Test ---")
rf_best = grid_rf.best_estimator_
y_pred_rf = rf_best.predict(X_test)
print(classification_report(y_test, y_pred_rf, target_names=le.classes_))


--- Đang chạy GridSearch cho Random Forest ---
Fitting 1 folds for each of 12 candidates, totalling 12 fits
Best RF params: {'rf__max_depth': None, 'rf__min_samples_leaf': 1, 'rf__min_samples_split': 10}
Best Dev F1-macro: 0.633106960950764

--- Kết quả Random Forest trên tập Test ---
              precision    recall  f1-score   support

    negative       0.77      0.74      0.76       151
     neutral       0.70      0.77      0.74       194
    positive       0.69      0.62      0.65       145

    accuracy                           0.72       490
   macro avg       0.72      0.71      0.71       490
weighted avg       0.72      0.72      0.72       490



In [9]:
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.metrics import confusion_matrix

def save_confusion_matrix(y_true, y_pred, classes, method_name):
    """
    Vẽ và lưu Confusion Matrix vào thư mục Confusion-Matrix/{method_name}/
    """
    # 1. Tạo đường dẫn thư mục: Confusion-Matrix/KNN hoặc Confusion-Matrix/RandomForest
    output_dir = os.path.join("Confusion-Matrix", method_name)
    os.makedirs(output_dir, exist_ok=True) # Tự tạo thư mục nếu chưa có
    
    # 2. Tính toán Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    
    # 3. Vẽ biểu đồ
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=classes, 
                yticklabels=classes)
    
    plt.title(f'Confusion Matrix - {method_name}')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.tight_layout()
    
    # 4. Lưu file
    save_path = os.path.join(output_dir, "confusion_matrix.png")
    plt.savefig(save_path)
    plt.close() # Đóng plot để giải phóng bộ nhớ
    
    print(f"Đã lưu Confusion Matrix cho {method_name} tại: {save_path}")

# --- Thực hiện lưu cho KNN ---
save_confusion_matrix(y_test, y_pred_knn, le.classes_, "KNN")

# --- Thực hiện lưu cho Random Forest ---
save_confusion_matrix(y_test, y_pred_rf, le.classes_, "RandomForest")

Đã lưu Confusion Matrix cho KNN tại: Confusion-Matrix\KNN\confusion_matrix.png
Đã lưu Confusion Matrix cho RandomForest tại: Confusion-Matrix\RandomForest\confusion_matrix.png


In [10]:
from sklearn.metrics import classification_report, accuracy_score


In [11]:
# Đánh giá trên tập Test
print("\n--- Kết quả Random Forest trên tập Test ---")
rf_best = grid_rf.best_estimator_
y_pred_rf = rf_best.predict(X_test)

acc_rf = accuracy_score(y_test, y_pred_rf)
print("Test Accuracy:", acc_rf)

print(classification_report(
    y_test,
    y_pred_rf,
    target_names=le.classes_
))



--- Kết quả Random Forest trên tập Test ---
Test Accuracy: 0.7183673469387755
              precision    recall  f1-score   support

    negative       0.77      0.74      0.76       151
     neutral       0.70      0.77      0.74       194
    positive       0.69      0.62      0.65       145

    accuracy                           0.72       490
   macro avg       0.72      0.71      0.71       490
weighted avg       0.72      0.72      0.72       490



In [12]:
# Đánh giá trên tập Test
print("\n--- Kết quả KNN trên tập Test ---")
knn_best = grid_knn.best_estimator_
y_pred_knn = knn_best.predict(X_test)

acc_knn = accuracy_score(y_test, y_pred_knn)
print("Test Accuracy:", acc_knn)

print(classification_report(
    y_test,
    y_pred_knn,
    target_names=le.classes_
))



--- Kết quả KNN trên tập Test ---
Test Accuracy: 0.6632653061224489
              precision    recall  f1-score   support

    negative       0.67      0.77      0.72       151
     neutral       0.66      0.69      0.68       194
    positive       0.66      0.52      0.58       145

    accuracy                           0.66       490
   macro avg       0.66      0.66      0.66       490
weighted avg       0.66      0.66      0.66       490



In [13]:
from sklearn.metrics import f1_score

In [14]:
# Đánh giá trên tập Test
print("\n--- Kết quả Random Forest trên tập Test ---")
rf_best = grid_rf.best_estimator_
y_pred_rf = rf_best.predict(X_test)

f1 = f1_score(y_test, y_pred_rf, average='weighted')
print("Test F1 Score:", f1)

print(classification_report(
    y_test,
    y_pred_rf,
    target_names=le.classes_
))



--- Kết quả Random Forest trên tập Test ---
Test F1 Score: 0.7173113193573553
              precision    recall  f1-score   support

    negative       0.77      0.74      0.76       151
     neutral       0.70      0.77      0.74       194
    positive       0.69      0.62      0.65       145

    accuracy                           0.72       490
   macro avg       0.72      0.71      0.71       490
weighted avg       0.72      0.72      0.72       490



In [15]:
# Đánh giá trên tập Test
print("\n--- Kết quả KNN trên tập Test ---")
knn_best = grid_knn.best_estimator_
y_pred_knn = knn_best.predict(X_test)

f1_knn = f1_score(y_test, y_pred_knn, average='weighted')
print("Test F1 Score:", f1_knn)

print(classification_report(
    y_test,
    y_pred_knn,
    target_names=le.classes_
))



--- Kết quả KNN trên tập Test ---
Test F1 Score: 0.6593112395747808
              precision    recall  f1-score   support

    negative       0.67      0.77      0.72       151
     neutral       0.66      0.69      0.68       194
    positive       0.66      0.52      0.58       145

    accuracy                           0.66       490
   macro avg       0.66      0.66      0.66       490
weighted avg       0.66      0.66      0.66       490

